# SimPro-ValueNetwork — From-Scratch and Three-Stage Training

Use fixed preprocessing assets and randomly initialize the network. The curriculum warms up at depth 2, trains independent linear-weight branches, then applies discounted supervision to each branch. Every phase creates a new Adam optimizer and cosine scheduler. This notebook neither loads nor modifies released model weights.

See [TRAINING.md](../docs/TRAINING.md) for definitions. Run [functional validation](colab_comprehensive_validation.ipynb) first. Production training requires labeled Train/Val data and sufficient compute time.

Optional learning example; pretrained inference does not require this notebook.

Extract the project to `/content/SimPro-ValueNetwork`, or set `PROJECT_DIR` to your project root. Google Drive is optional: enable `USE_GOOGLE_DRIVE` and select your mounted project path. Checkpoints may be stored at any readable location.


The project is SimPro-ValueNetwork; the distribution/import name is simpro_vn, with simpro-vn and simpro-vn-train console entrypoints. Use normal installation in a fresh runtime. Use [Comprehensive Validation](colab_comprehensive_validation.ipynb) for all functional checks in one run.


In [ ]:
USE_GOOGLE_DRIVE = False  # Enable only if your files are stored in Google Drive.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
from uuid import uuid4
import json
import subprocess
import sys

PROJECT_DIR = Path('/content/SimPro-ValueNetwork')
assert (PROJECT_DIR / 'pyproject.toml').is_file(), PROJECT_DIR
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8]

## Installation

Select a Colab GPU runtime for GPU execution and confirm CUDA-enabled PyTorch. Restart the runtime after updating an imported package.


In [ ]:
%pip install "{PROJECT_DIR}[chemistry]"
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## Configuration

The default input is the included 300-row smoke CSV. SMOKE_RUN uses one epoch per phase but retains the production-size network and the entire selected dataset. Set it to False for the full schedule; use the validation notebook's tiny models for code checks. MODE='scratch' runs a single depth and selected schedule.


In [ ]:
from simpro_vn.training import TrainingConfig, run_training

DATASET = PROJECT_DIR / 'examples/data/value_network_smoke.csv'
MODE = 'curriculum'  # 'curriculum' or 'scratch'
SMOKE_RUN = True
config = TrainingConfig(
    dataset_path=str(DATASET),
    output_dir=str(Path('/content/training_runs') / ('training_' + RUN_ID)),
    mode=MODE, representation='smiles', min_frequency=5000, depth=6,
    epochs=1 if SMOKE_RUN else 10,
    warmup_epochs=1 if SMOKE_RUN else 2,
    linear_epochs=1 if SMOKE_RUN else 10,
    discounted_epochs=1 if SMOKE_RUN else 10,
    curriculum_depths=(2, 4, 6),
    batch_size=8, stage3_batch_size=8,
    accumulation_steps=1, learning_rate=1e-4, schedule='linear',
    d_model=768, nhead=12, dim_feedforward=3072, max_seq_len=128,
    precision='fp32', device='auto',
)
assert DATASET.is_file(), DATASET
print(config)

## Run and Outputs

Weights are written only inside a new directory. Each curriculum depth starts from the same warm-up best; its linear best initializes the discounted phase. This example sets stage3 batch size to 8; adjust for available memory. Changes to training settings do not guarantee reproduction of released model values.


In [ ]:
result = run_training(config)
print(json.dumps(result, indent=2))

## Load Results

Every phase produces best.pt, last.pt, history.json, and a loadable inference_config.json. Select depth 6's final model below; update SELECTED_DEPTH if changing curriculum_depths.


In [ ]:
from simpro_vn import ValueNetworkPredictor

SELECTED_DEPTH = 6
selected = result['branches'][str(SELECTED_DEPTH)]['discounted'] if MODE == 'curriculum' else result
predictor = ValueNetworkPredictor.from_config(Path(selected['directory']) / 'inference_config.json')
print(predictor.predict('CCO'))
print('Training output:', config.output_dir)

## Persistence and Interruption

Before ending the runtime, copy the entire output directory to a new Drive location. Individual phases can resume from last.pt as described in TRAINING.md; the full curriculum does not automatically continue remaining phases. Use the phase_config.json and a new output_dir when resuming a phase.


In [ ]:
import shutil

EXPORT_RESULTS = False
EXPORT_ROOT = None  # Set a persistent directory, e.g. a mounted Drive folder.
if EXPORT_RESULTS:
    if EXPORT_ROOT is None:
        raise ValueError('Set EXPORT_ROOT before exporting results')
    destination = Path(EXPORT_ROOT).expanduser() / RUN_ID
    shutil.copytree(config.output_dir, destination)
    print('Saved:', destination)
